# 03 - Modelling the approval decision

Target: `loan_status` (1 = approved) in the **synthetic** primary dataset. This is a pre-screening prototype; it does **not** predict default, and the scores below say how well the model reproduces past synthetic approval decisions, not real-world accuracy.

Method (CLAUDE.md section 9): stratified 5-fold CV with every scaler, encoder and calibrator fitted inside the fold (sklearn Pipelines), fixed seeds, and the rule "pick the simplest model within about 1% of the best". Gender and marital status are not in the data and are blocked by `model.make_estimator`.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.calibration import calibration_curve

from loanlens import cleaning
from loanlens import model as m
from loanlens.config import load_policy

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
pd.set_option("display.width", 160)

df = pd.read_csv(cleaning.PROCESSED_PATH)
frame = m.build_frame(df)
y = frame[m.TARGET]
policy = load_policy()
MIN_CIBIL = policy.rules.min_cibil.value
gate = (frame["cibil_score"] >= MIN_CIBIL).to_numpy()  # rows the hard rule lets through
print(
    f"{len(frame):,} rows, approval share {y.mean():.1%}; "
    f"{gate.sum():,} rows pass the min_cibil rule ({MIN_CIBIL:.0f})"
)

4,269 rows, approval share 62.2%; 2,484 rows pass the min_cibil rule (550)


## 1. Feature sets

- **FULL**: every cleaned column (including education, self-employed, dependents) plus `loan_to_income`.
- **RAW**: every cleaned column, no derived ratio.
- **COMPACT**: `cibil_score`, `loan_term`, `loan_to_income`, `income_annum`, `total_assets`. Drops the variables EDA showed have no effect and leaves out raw `loan_amount` (the loan enters only through `loan_to_income`; see section 4 for why).

In [2]:
RAW = [*m.RAW_NUMERIC, *m.CATEGORICAL]
FULL = list(m.FULL_FEATURES)
COMPACT = list(m.FINAL_FEATURES)
KINDS = ["logreg", "random_forest", "xgboost"]
METRICS = ["roc_auc", "pr_auc", "precision", "recall", "f1", "accuracy", "brier"]


def cv(name, kind, cols, monotone=None, calibration=None):
    est = m.make_estimator(kind, cols, monotone)
    if calibration:
        est = m.calibrated(est, calibration)
    return m.cross_validate(name, est, frame[list(cols)], y)


def table(results, cols=METRICS):
    return m.summary_table(results)[cols + ["f1_std"]]

## 2. Model comparison (stratified 5-fold CV)

In [3]:
comparison = []
for fs_name, cols in [("RAW", RAW), ("FULL", FULL), ("COMPACT", COMPACT)]:
    for kind in KINDS:
        comparison.append(cv(f"{kind} | {fs_name}", kind, cols))
table(comparison)

,roc_auc,pr_auc,precision,recall,f1,accuracy,brier,f1_std
model,,,,,,,,
logreg | RAW,0.9681,0.9803,0.9324,0.9334,0.9328,0.9164,0.0637,0.0063
random_forest | RAW,0.9977,0.9984,0.9822,0.9925,0.9873,0.9841,0.0167,0.0031
xgboost | RAW,0.9976,0.9982,0.9858,0.9902,0.9880,0.9850,0.0119,0.0011
logreg | FULL,0.9687,0.9799,0.9302,0.9322,0.9312,0.9143,0.0632,0.0055
random_forest | FULL,0.9985,0.9988,0.9940,1.0000,0.9970,0.9963,0.0063,0.0016
xgboost | FULL,0.9993,0.9996,0.9925,0.9985,0.9955,0.9944,0.0048,0.0016
logreg | COMPACT,0.9689,0.9802,0.9307,0.9337,0.9322,0.9154,0.0630,0.0047
random_forest | COMPACT,0.9989,0.9992,0.9940,1.0000,0.9970,0.9963,0.0040,0.0016
xgboost | COMPACT,0.9992,0.9995,0.9925,0.9989,0.9957,0.9946,0.0051,0.0014


Confusion matrices (out-of-fold predictions at threshold 0.5; rows = actual, columns = predicted).

In [4]:
for r in comparison:
    cm = pd.DataFrame(
        r.confusion,
        index=["actual rejected", "actual approved"],
        columns=["pred rejected", "pred approved"],
    )
    print(r.name)
    print(cm.to_string(), end="\n\n")

logreg | RAW
                 pred rejected  pred approved
actual rejected           1433            180
actual approved            177           2479

random_forest | RAW
                 pred rejected  pred approved
actual rejected           1565             48
actual approved             20           2636

xgboost | RAW
                 pred rejected  pred approved
actual rejected           1575             38
actual approved             26           2630

logreg | FULL
                 pred rejected  pred approved
actual rejected           1427            186
actual approved            180           2476

random_forest | FULL
                 pred rejected  pred approved
actual rejected           1597             16
actual approved              0           2656

xgboost | FULL
                 pred rejected  pred approved
actual rejected           1593             20
actual approved              4           2652

logreg | COMPACT
                 pred rejected  pred approved
actual

**Reading.** Logistic regression sits around 0.97 ROC-AUC: a linear score cannot express the sharp CIBIL step plus the short-term interaction. The two tree ensembles reach about 0.999. Adding `loan_to_income` lifts the trees (RAW vs FULL), and dropping education, self-employment, dependents and the individual asset columns costs nothing (FULL vs COMPACT).

## 3. Why are the scores near-perfect? Leakage and ablation checks

Ablations use XGBoost on the COMPACT set (unconstrained).

In [5]:
base = [c for c in COMPACT]
ablations = [
    cv("all COMPACT features", "xgboost", base),
    cv("CIBIL only", "xgboost", ["cibil_score"]),
    cv("without CIBIL", "xgboost", [c for c in base if c != "cibil_score"]),
    cv("without loan_term", "xgboost", [c for c in base if c != "loan_term"]),
    cv("without loan_to_income", "xgboost", [c for c in base if c != "loan_to_income"]),
    cv("CIBIL + loan_term only", "xgboost", ["cibil_score", "loan_term"]),
]
abl = table(ablations, ["roc_auc", "pr_auc", "f1", "accuracy"])
abl["roc_auc_drop_vs_all"] = abl["roc_auc"].iloc[0] - abl["roc_auc"]
abl

,roc_auc,pr_auc,f1,accuracy,f1_std,roc_auc_drop_vs_all
model,,,,,,
all COMPACT features,0.9992,0.9995,0.9957,0.9946,0.0014,0.0000
CIBIL only,0.9603,0.9789,0.9607,0.9527,0.0046,0.0388
without CIBIL,0.5837,0.7379,0.6980,0.5821,0.0055,0.4155
without loan_term,0.9822,0.9906,0.9547,0.9447,0.0032,0.0170
without loan_to_income,0.9948,0.9968,0.9663,0.9581,0.0073,0.0044
CIBIL + loan_term only,0.9912,0.9921,0.9641,0.9553,0.0055,0.0079


In [6]:
# Sanity check: with shuffled labels the pipeline must fall to chance.
rng = np.random.default_rng(m.SEED)
y_shuffled = pd.Series(rng.permutation(y.to_numpy()), index=y.index)
shuf = m.cross_validate(
    "shuffled labels", m.make_estimator("xgboost", COMPACT), frame[COMPACT], y_shuffled
)
print(f"Shuffled-label ROC-AUC: {shuf.mean['roc_auc']:.3f} (chance = 0.5)")

# Other leakage checks
print("loan_id in features:", "loan_id" in FULL)
print("Duplicate feature rows:", int(frame[FULL].duplicated().sum()))
print(
    "Derived features use only applicant inputs (no target): "
    "loan_to_income = loan_amount / income_annum, total_assets = sum of assets"
)
corr = frame[COMPACT + ["loan_status"]].corr()["loan_status"].drop("loan_status")
print("\nCorrelation with the label:")
print(corr.sort_values(key=abs, ascending=False).to_string())

Shuffled-label ROC-AUC: 0.499 (chance = 0.5)
loan_id in features: False
Duplicate feature rows: 0
Derived features use only applicant inputs (no target): loan_to_income = loan_amount / income_annum, total_assets = sum of assets

Correlation with the label:
cibil_score       0.7705
loan_term        -0.1130
loan_to_income    0.0877
income_annum     -0.0152
total_assets     -0.0113


In [7]:
fitted = m.make_estimator("xgboost", COMPACT).fit(frame[COMPACT], y)
gain = pd.Series(fitted[-1].get_booster().get_score(importance_type="gain"))
(gain / gain.sum()).sort_values(ascending=False).to_frame("share_of_total_gain")

,share_of_total_gain
cibil_score,0.7430
loan_term,0.1341
loan_to_income,0.1056
total_assets,0.0097
income_annum,0.0077


**Findings.**

- No leakage found: no identifier or target-derived column is used, there are no duplicate rows across folds, and shuffled labels drop to chance.
- The near-perfect score is because the synthetic label is almost a deterministic rule. CIBIL alone gets about 0.96 ROC-AUC, and without CIBIL the model is close to chance. `loan_term` and `loan_to_income` supply the remaining accuracy, through the low-CIBIL short-term quirk.
- **Suspicious behaviour:** the unconstrained model learns that for low-CIBIL, short-term applicants a *bigger* loan relative to income raises approval odds. That contradicts lending sense, so section 4 tests constraints against it.

## 4. Monotonic constraints (milestone 1 review, decision 4)

Test: does forcing approval to be non-decreasing in CIBIL (and non-increasing in loan-to-income) cost less than 1%? The *loan-size probe* raises every loan by 25% (income fixed) on a model fitted to all data and reports how often the approval probability goes **up**. Good behaviour is 0%.

In [8]:
WITH_LOAN = [
    "cibil_score",
    "loan_term",
    "income_annum",
    "loan_amount",
    "loan_to_income",
    "total_assets",
]
low_short = frame[(frame["cibil_score"] <= MIN_CIBIL) & frame["loan_term"].isin([2, 4])]
variants = [
    ("with raw loan_amount | none", WITH_LOAN, None),
    ("with raw loan_amount | CIBIL+", WITH_LOAN, {"cibil_score": 1}),
    ("with raw loan_amount | CIBIL+, LTI-", WITH_LOAN, {"cibil_score": 1, "loan_to_income": -1}),
    ("COMPACT | none", COMPACT, None),
    ("COMPACT | CIBIL+", COMPACT, {"cibil_score": 1}),
    ("COMPACT | CIBIL+, LTI-", COMPACT, {"cibil_score": 1, "loan_to_income": -1}),
]
mono_rows, mono_results = [], {}
for name, cols, mono in variants:
    r = cv(name, "xgboost", cols, mono)
    mono_results[name] = r
    fit = m.make_estimator("xgboost", cols, mono).fit(frame[cols], y)
    probe_all = m.loan_size_probe(fit, frame, cols)
    probe_low = m.loan_size_probe(fit, low_short, cols)
    gated = m.classification_metrics(y[gate], r.oof_proba[gate])
    system = m.classification_metrics(y, np.where(gate, r.oof_proba, 0.0))
    mono_rows.append(
        {
            "variant": name,
            "roc_auc": r.mean["roc_auc"],
            "f1": r.mean["f1"],
            "gated_f1": gated["f1"],
            "system_f1": system["f1"],
            "system_accuracy": system["accuracy"],
            "loan+25%: share up (all)": probe_all["share_increased"],
            "loan+25%: share up (low CIBIL, short)": probe_low["share_increased"],
            "loan+25%: mean change (low CIBIL, short)": probe_low["mean_change"],
        }
    )
mono_table = pd.DataFrame(mono_rows).set_index("variant")
mono_table

,roc_auc,f1,gated_f1,system_f1,system_accuracy,loan+25%: share up (all),"loan+25%: share up (low CIBIL, short)","loan+25%: mean change (low CIBIL, short)"
variant,,,,,,,,
with raw loan_amount | none,0.9991,0.9957,0.9968,0.9609,0.9529,0.4938,0.4774,0.2395
with raw loan_amount | CIBIL+,0.9988,0.9959,0.9970,0.9611,0.9532,0.3865,0.5452,0.2348
"with raw loan_amount | CIBIL+, LTI-",0.9984,0.9914,0.9970,0.9611,0.9532,0.5406,0.8475,0.2184
COMPACT | none,0.9992,0.9957,0.9968,0.9609,0.9529,0.5355,0.4124,0.2397
COMPACT | CIBIL+,0.9986,0.9961,0.9970,0.9611,0.9532,0.4214,0.6554,0.2369
"COMPACT | CIBIL+, LTI-",0.9952,0.9677,0.9970,0.9611,0.9532,0.0000,0.0000,-0.0012


Columns: `gated_f1` = F1 on the rows that pass the enforced `min_cibil` rule (the only rows on which the model decides); `system_*` = the full decision (enforced CIBIL rule, then the model) against the labels.

In [9]:
ref, con = mono_table.loc["COMPACT | none"], mono_table.loc["COMPACT | CIBIL+, LTI-"]
for metric in ["roc_auc", "f1", "gated_f1", "system_f1", "system_accuracy"]:
    print(
        f"{metric:16s}: unconstrained {ref[metric]:.4f}  constrained "
        f"{con[metric]:.4f}  cost {100 * (ref[metric] - con[metric]):+.2f} points"
    )
cibil_only = mono_table.loc["COMPACT | CIBIL+"]
print(
    f"\nCIBIL-only constraint: ROC-AUC cost "
    f"{100 * (ref['roc_auc'] - cibil_only['roc_auc']):+.2f} points, but the "
    f"loan-size probe still rises for "
    f"{cibil_only['loan+25%: share up (low CIBIL, short)']:.0%} of low-CIBIL short-term"
    ' rows'
)

roc_auc         : unconstrained 0.9992  constrained 0.9952  cost +0.40 points
f1              : unconstrained 0.9957  constrained 0.9677  cost +2.79 points
gated_f1        : unconstrained 0.9968  constrained 0.9970  cost -0.02 points
system_f1       : unconstrained 0.9609  constrained 0.9611  cost -0.02 points
system_accuracy : unconstrained 0.9529  constrained 0.9532  cost -0.02 points

CIBIL-only constraint: ROC-AUC cost +0.06 points, but the loan-size probe still rises for 66% of low-CIBIL short-term rows


**Reading.**

- A CIBIL constraint on its own is almost free, but it does **not** stop the loan-to-income quirk. That needs a non-increasing constraint on `loan_to_income`.
- With raw `loan_amount` among the features, the `loan_to_income` constraint is **bypassed**: the model rebuilds the quirk from `loan_amount` and income, and the probe still rises. Hence the COMPACT set leaves raw `loan_amount` out.
- On COMPACT, the CIBIL+ / LTI- constraints remove the quirk (probe share = 0%). They cost under 1% of ROC-AUC. F1 against the raw labels drops by a few points, but that loss sits entirely in CIBIL < 550 rows, which the enforced `min_cibil` rule rejects anyway. On the rows the model actually decides, and for the full rule + model decision, the cost is under 0.1 points. **So the constrained model is used.**

## 5. Calibration

In [10]:
MONO = dict(m.FINAL_MONOTONE)
cal = {
    "uncalibrated": mono_results["COMPACT | CIBIL+, LTI-"],
    "sigmoid (Platt)": cv("sigmoid", "xgboost", COMPACT, MONO, "sigmoid"),
    "isotonic": cv("isotonic", "xgboost", COMPACT, MONO, "isotonic"),
}
cal_table = pd.DataFrame(
    {k: v.mean[["roc_auc", "f1", "brier", "log_loss"]] for k, v in cal.items()}
).T
cal_table

,roc_auc,f1,brier,log_loss
uncalibrated,0.9952,0.9677,0.0275,0.0868
sigmoid (Platt),0.9952,0.9683,0.0310,0.1143
isotonic,0.9942,0.9639,0.0257,0.0803


In [11]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[0, 1], y=[0, 1], mode="lines", name="perfect", line={"dash": "dash", "color": "grey"}
    )
)
for name, r in cal.items():
    frac_pos, mean_pred = calibration_curve(y, r.oof_proba, n_bins=10, strategy="quantile")
    fig.add_trace(go.Scatter(x=mean_pred, y=frac_pos, mode="lines+markers", name=name))
fig.update_layout(
    title="Calibration curve (out-of-fold, quantile bins)",
    xaxis_title="Mean predicted approval probability",
    yaxis_title="Observed approval rate",
)
fig

In [12]:
best_cal = cal_table["brier"].idxmin()
print(f"Lowest Brier score: {best_cal}")
print(f"Configured in model.py: {m.FINAL_CALIBRATION}")

Lowest Brier score: isotonic
Configured in model.py: isotonic


## 6. Model choice: simplest within about 1% of the best

Candidates are the constrained versions of each model on COMPACT (logistic regression is monotone by construction). Primary metric: ROC-AUC. Simplicity order: logistic regression < XGBoost (300 trees of depth 4) < random forest (300 fully grown trees); the leaf counts below support that order.

In [13]:
cands = {
    "logreg": cv("logreg", "logreg", COMPACT),
    "xgboost": cal["isotonic"],
    "random_forest": cv("random_forest", "random_forest", COMPACT, MONO, "isotonic"),
}
rf_fit = m.make_estimator("random_forest", COMPACT, MONO).fit(frame[COMPACT], y)
xgb_fit = m.make_estimator("xgboost", COMPACT, MONO).fit(frame[COMPACT], y)
leaves = {
    "logreg": len(COMPACT) + 1,
    "xgboost": int((xgb_fit[-1].get_booster().trees_to_dataframe()["Feature"] == "Leaf").sum()),
    "random_forest": int(sum(t.get_n_leaves() for t in rf_fit[-1].estimators_)),
}
choice = pd.DataFrame(
    {
        k: {
            **v.mean[["roc_auc", "pr_auc", "f1", "brier"]].to_dict(),
            "parameters_or_leaves": leaves[k],
        }
        for k, v in cands.items()
    }
).T
best = choice["roc_auc"].max()
choice["within_1pct_of_best"] = choice["roc_auc"] >= best - 0.01
choice

,roc_auc,pr_auc,f1,brier,parameters_or_leaves,within_1pct_of_best
logreg,0.9689,0.9802,0.9322,0.0630,6.0000,False
xgboost,0.9942,0.9946,0.9639,0.0257,2964.0000,True
random_forest,0.9942,0.9942,0.9647,0.0243,30491.0000,True


In [14]:
order = ["logreg", "xgboost", "random_forest"]
chosen = next(k for k in order if choice.loc[k, "within_1pct_of_best"])
print(f"Chosen: {chosen}")
assert chosen == m.FINAL_KIND, "model.py FINAL_KIND must match the notebook choice"

Chosen: xgboost


## 7. Train the final model on all data and save it

In [15]:
final_cv = cal["isotonic"]
cv_metrics = {k: round(float(v), 4) for k, v in final_cv.mean.items()}
artifact = m.train_final(df, cv_metrics=cv_metrics)
path = m.save_artifact(artifact)
print(
    f"Saved {path.name}: version {artifact['version']}, {artifact['kind']}, "
    f"calibration {artifact['calibration']}"
)
print("Features:", artifact["features"])
print("Monotone constraints:", artifact["monotone"])
pd.DataFrame({"mean": final_cv.mean, "std": final_cv.std})

Saved loanlens_model.joblib: version 1.0.0, xgboost, calibration isotonic
Features: ['cibil_score', 'loan_term', 'loan_to_income', 'income_annum', 'total_assets']
Monotone constraints: {'cibil_score': 1, 'loan_to_income': -1}


,mean,std
roc_auc,0.9942,0.0020
pr_auc,0.9946,0.0019
precision,0.9527,0.0136
recall,0.9759,0.0203
f1,0.9639,0.0058
accuracy,0.9546,0.0069
brier,0.0257,0.0032
log_loss,0.0803,0.0120


In [16]:
print("Out-of-fold confusion matrix (final model, threshold 0.5):")
print(
    pd.DataFrame(
        final_cv.confusion,
        index=["actual rejected", "actual approved"],
        columns=["pred rejected", "pred approved"],
    ).to_string()
)
system = m.classification_metrics(y, np.where(gate, final_cv.oof_proba, 0.0))
print("\nFull decision (enforced min_cibil rule, then model) vs labels:")
print({k: round(v, 4) for k, v in system.items()})

Out-of-fold confusion matrix (final model, threshold 0.5):
                 pred rejected  pred approved
actual rejected           1483            130
actual approved             64           2592

Full decision (enforced min_cibil rule, then model) vs labels:
{'roc_auc': np.float64(0.9635), 'pr_auc': np.float64(0.972), 'precision': np.float64(0.9948), 'recall': np.float64(0.9292), 'f1': np.float64(0.9609), 'accuracy': 0.9529, 'brier': np.float64(0.0473), 'log_loss': 0.6186}


## Conclusion

**Final model:** XGBoost (300 trees, depth 4) on the COMPACT features (`cibil_score`, `loan_term`, `loan_to_income`, `income_annum`, `total_assets`). Monotone constraints: approval can only rise with CIBIL and only fall with loan-to-income. Probabilities are isotonic-calibrated. Saved to `models/loanlens_model.joblib` with its feature list and version.

**Why this model**

- Logistic regression (ROC-AUC 0.969) is more than 1% behind the best (0.999), so it does not qualify, even though it is the simplest.
- Constrained XGBoost and constrained random forest score the same (ROC-AUC 0.994). XGBoost is the smaller model (about 3,000 leaves vs about 30,000), so it is the simpler of the two that qualify.
- The constraints cost 0.40 ROC-AUC points against the best unconstrained model (under the 1% limit). F1 against the raw labels falls from 0.996 to 0.964, but the whole loss is in CIBIL < 550 rows, which the enforced `min_cibil` rule rejects regardless. On the rows the model actually decides, and for the full rule + model decision, the constrained model is as good as or better than the unconstrained one (system F1 0.961 vs 0.961).
- Isotonic calibration gives the lowest Brier score and log loss.

**What the ablations show**

- CIBIL alone: ROC-AUC 0.960. Without CIBIL: 0.584, close to chance. CIBIL is about 74% of the model's total gain.
- Without `loan_term`: ROC-AUC falls by 1.7 points. Without `loan_to_income`: by 0.4 points. Both matter only through the synthetic low-CIBIL short-term quirk.
- With shuffled labels, ROC-AUC is 0.499. No leakage was found.

**Limitations**

- The near-perfect scores reflect how simple the synthetic labelling rule is, not real-world performance.
- The model is constrained *against* part of the training labels (the high-LTI low-CIBIL approvals). That is a deliberate choice for sound lending logic, so it disagrees with some historical labels.
